# Statistics for repeated-seed tutor evaluations

This notebook uses 24 tasks, 8 seeds, four tutors, and a no-tutor control. The scientific question is: **across tasks like these, how much does each tutor change the score relative to no tutor?** Therefore the independent unit and bootstrap resampling unit are tasks—not generated sessions.

Generate the data first from the repository root with `npm run eval:statistics:generate`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import bootstrap, spearmanr

HERE = Path.cwd()
csv_path = HERE / "results.csv"
if not csv_path.exists():
    csv_path = HERE / "evals" / "statistics-tutorial" / "results.csv"
df = pd.read_csv(csv_path)
print(df.shape)
df.head()

(960, 11)


,task_id,difficulty,seed,condition,score,passed,simulator_score,human_score,predicted_failure,annotator_a_failure,annotator_b_failure
0,task_01,24.003,0,no_tutor,71.236,1,67.230,74.319,0,0,0
1,task_01,24.003,0,socratic,82.698,1,98.842,85.188,0,0,0
2,task_01,24.003,0,worked_example,83.355,1,63.140,75.794,0,0,0
3,task_01,24.003,0,hint_ladder,80.613,1,73.766,76.432,0,0,0
4,task_01,24.003,0,reflective,73.939,1,78.466,83.907,0,0,0


## 1. Experimental hierarchy and estimand

A row is a session, but rows sharing a task inherit the same difficulty and task-specific tutor response. Seeds measure run-to-run randomness *inside* a task. They improve the estimate for that task; they do not expand the population of tasks.

Our estimand is the macro-average task effect: for each task, average over seeds, subtract that task's no-tutor score, then average those 24 paired differences. Every task receives equal weight.

In [2]:
wide = df.pivot(index=["task_id", "seed"], columns="condition", values="score")
tutors = [c for c in wide.columns if c != "no_tutor"]

# Seed-level paired differences, followed by task aggregation.
seed_deltas = wide[tutors].sub(wide["no_tutor"], axis=0)
task_deltas = seed_deltas.groupby(level="task_id").mean()

assert len(task_deltas) == 24
effects = task_deltas.mean().sort_values(ascending=False)
effects

condition
socratic          6.388396
hint_ladder       6.011266
worked_example    4.665964
reflective        2.814313
dtype: float64

## 2. Correct clustered bootstrap versus pseudo-replication

The correct bootstrap samples 24 task effects with replacement. The misleading version samples all 24 × 8 task-seed differences as though they were independent. Both retain the tutor/control pairing; only the first respects the independence structure.

`paired=True` in SciPy is crucial when passing tutor and control as separate arrays because it resamples the same indices from both. Here we bootstrap an already-computed difference, which is algebraically equivalent.

In [3]:
task_scores = df.groupby(["task_id", "condition"])["score"].mean().unstack()

def mean_difference(tutor, control, axis=-1):
    return np.mean(tutor - control, axis=axis)

def paired_ci(tutor, control, seed=7):
    result = bootstrap(
        (np.asarray(tutor), np.asarray(control)), mean_difference,
        paired=True,
        n_resamples=20_000, confidence_level=0.95,
        method="BCa", rng=np.random.default_rng(seed)
    )
    return result.confidence_interval.low, result.confidence_interval.high, result.standard_error

comparison = []
for tutor in tutors:
    clustered = paired_ci(task_scores[tutor], task_scores["no_tutor"]) # n = 24 tasks
    naive = paired_ci(wide[tutor], wide["no_tutor"])                  # falsely n = 192
    comparison.append({
        "tutor": tutor,
        "mean_effect": task_deltas[tutor].mean(),
        "cluster_low": clustered[0], "cluster_high": clustered[1],
        "cluster_width": clustered[1] - clustered[0],
        "naive_low": naive[0], "naive_high": naive[1],
        "naive_width": naive[1] - naive[0],
    })

ci_table = pd.DataFrame(comparison).set_index("tutor")
ci_table["naive/cluster_width"] = ci_table["naive_width"] / ci_table["cluster_width"]
ci_table.round(3)

,mean_effect,cluster_low,cluster_high,cluster_width,naive_low,naive_high,naive_width,naive/cluster_width
tutor,,,,,,,,
hint_ladder,6.011,4.527,7.639,3.112,5.085,6.940,1.855,0.596
reflective,2.814,1.068,4.991,3.924,1.734,3.920,2.186,0.557
socratic,6.388,4.341,8.443,4.103,5.274,7.487,2.213,0.539
worked_example,4.666,3.005,6.414,3.409,3.669,5.659,1.990,0.584


If seeds within a task were perfectly correlated, eight seeds would still provide only one independent task observation. If they were completely independent conditional on task, they would estimate that task's mean very precisely—but generalization to new tasks would still depend primarily on the number and diversity of tasks. To estimate both levels explicitly, use a hierarchical model or a two-stage/cluster bootstrap aligned to the target estimand.

## 3. Confusion matrix, precision, and recall

We treat `human_score < 60` as actual failure and ask whether the simulator flags failure. Precision answers: *of sessions flagged, how many truly failed?* Recall answers: *of true failures, how many were flagged?* Always state which class is positive.

In [4]:
actual = (df["human_score"] < 60).astype(int)
predicted = df["predicted_failure"].astype(int)
cm = pd.crosstab(actual, predicted, rownames=["actual"], colnames=["predicted"], dropna=False).reindex(index=[0,1], columns=[0,1], fill_value=0)
tn, fp, fn, tp = cm.to_numpy().ravel()
precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
print(cm)
print({"precision_failure": precision, "recall_failure": recall})

predicted    0    1
actual             
0          432   50
1           66  412
{'precision_failure': np.float64(0.8917748917748918), 'recall_failure': np.float64(0.8619246861924686)}


## 4. Cohen's kappa for two annotators

Raw agreement can be high merely because both annotators usually choose the common class. Cohen's κ corrects observed agreement by the agreement expected from their marginal label frequencies: `κ = (p_observed - p_expected) / (1 - p_expected)`. A κ of 1 is perfect agreement; 0 is chance-level under the marginal model; negative values indicate worse-than-expected agreement. Interpret κ alongside the confusion table and class prevalence.

In [5]:
a = df["annotator_a_failure"].to_numpy()
b = df["annotator_b_failure"].to_numpy()
p_observed = np.mean(a == b)
p_a1, p_b1 = np.mean(a == 1), np.mean(b == 1)
p_expected = p_a1 * p_b1 + (1 - p_a1) * (1 - p_b1)
kappa = (p_observed - p_expected) / (1 - p_expected)
print(pd.crosstab(a, b, rownames=["annotator A"], colnames=["annotator B"]))
print({"observed_agreement": p_observed, "expected_agreement": p_expected, "cohen_kappa": kappa})

annotator B    0    1
annotator A          
0            381  113
1             99  367
{'observed_agreement': np.float64(0.7791666666666667), 'expected_agreement': np.float64(0.5), 'cohen_kappa': np.float64(0.5583333333333333)}


## 5. Spearman correlation for simulator–human rankings

Spearman's ρ is Pearson correlation applied to ranks. It asks whether the association is monotonic, not whether scores are numerically calibrated. Here we compare rankings across the four tutors. With only four items, the estimate is coarse and inference is weak; task-level analysis offers more observations but answers a different question.

In [6]:
tutor_rows = df[df["condition"] != "no_tutor"]
ranking = tutor_rows.groupby("condition")[["human_score", "simulator_score"]].mean()
rho_tutors = spearmanr(ranking["human_score"], ranking["simulator_score"])

task_ranking = df.groupby("task_id")[["human_score", "simulator_score"]].mean()
rho_tasks = spearmanr(task_ranking["human_score"], task_ranking["simulator_score"])
print(ranking.sort_values("human_score", ascending=False))
print("Across four tutors:", rho_tutors)
print("Across 24 tasks (different estimand):", rho_tasks)

                human_score  simulator_score
condition                                   
hint_ladder       62.240396        61.333672
socratic          62.072802        60.162781
worked_example    60.749786        60.181948
reflective        59.040333        58.096365
Across four tutors: SignificanceResult(statistic=np.float64(0.7999999999999999), pvalue=np.float64(0.20000000000000007))
Across 24 tasks (different estimand): SignificanceResult(statistic=np.float64(0.9991304347826085), pvalue=np.float64(7.374248237784458e-32))


## 6. Floor and ceiling diagnostics

A continuous score at its lower or upper bound loses resolution; a binary pass rate near 0 or 1 has little room to show worsening or improvement. Check these before interpreting a null effect. A tutor cannot visibly improve a task on which nearly everyone already passes.

In [7]:
bounds = df.groupby("condition").agg(
    floor_share=("score", lambda s: np.mean(s <= 5)),
    ceiling_share=("score", lambda s: np.mean(s >= 95)),
    pass_rate=("passed", "mean"),
)
task_condition_pass = df.groupby(["task_id", "condition"])["passed"].mean()
extreme_cells = task_condition_pass[(task_condition_pass <= 0.05) | (task_condition_pass >= 0.95)]
print(bounds)
print(f"Task-condition cells near floor/ceiling: {len(extreme_cells)} of {len(task_condition_pass)}")
extreme_cells.head(12)

                floor_share  ceiling_share  pass_rate
condition                                            
hint_ladder        0.000000       0.078125   0.546875
no_tutor           0.000000       0.036458   0.406250
reflective         0.005208       0.062500   0.473958
socratic           0.000000       0.078125   0.557292
worked_example     0.000000       0.057292   0.505208
Task-condition cells near floor/ceiling: 86 of 120


task_id  condition     
task_01  hint_ladder       1.0
         socratic          1.0
         worked_example    1.0
task_02  hint_ladder       0.0
         no_tutor          0.0
         reflective        0.0
         socratic          0.0
         worked_example    0.0
task_03  hint_ladder       1.0
         socratic          1.0
         worked_example    1.0
task_04  hint_ladder       1.0
Name: passed, dtype: float64

## What to report

1. Define the population and estimand before calculating anything.
2. Report 24 independent tasks and 8 repeated seeds per task—not `n = 192` independent examples.
3. Give each tutor's paired mean task effect and task-clustered 95% CI.
4. Describe task sampling, seed policy, missing runs, and aggregation.
5. For classifiers, state the positive class and report the full confusion matrix with precision and recall.
6. For annotators, report prevalence, raw agreement, κ, and disagreements.
7. For rankings, state exactly what was ranked and report Spearman ρ; do not call it calibration.
8. Inspect floor/ceiling behavior and task-level heterogeneity rather than relying only on a grand mean.